# Retrieval-Vergleich: BM25S vs. Embeddings

Gleiches Korpus, gleiche Queries, zwei Retrieval-Verfahren:

- **BM25S** – term-based, zaehlt Wortueberlappung zwischen Query und Chunk (lexikalisch)
- **Embeddings** – dense, vergleicht Vektoren im semantischen Raum

Korpus: `AfD_Parteiprogramm2026.txt`, gechunkt nach Saetzen.

In [ ]:
# Einmalig installieren, falls noch nicht vorhanden:
# %pip install bm25s PyStemmer sentence-transformers pandas numpy

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

TXT_PATH = Path("AfD_Parteiprogramm2026.txt")
TOP_K = 5
MIN_CHARS = 40   # sehr kurze Fragmente rausfiltern

## 1. Laden und Chunking

Wichtig: der Text kommt aus einem PDF und hat harte Zeilenumbrueche mitten im Satz.
Deshalb erst die Zeilenumbrueche zusammenfuehren, dann nach `.` splitten. Sonst
bekommst du Fragmente wie `'Die Klimaschutzpolitik der'` als eigene Chunks.

In [ ]:
raw = TXT_PATH.read_text(encoding="utf-8")

# Zeilenumbrueche und Mehrfach-Leerzeichen zu einem Fliesstext zusammenziehen
flat = " ".join(raw.split())

chunks = [c.strip() for c in flat.split(".")]
chunks = [c for c in chunks if len(c) >= MIN_CHARS]

print(f"Zeichen gesamt : {len(raw):,}")
print(f"Chunks         : {len(chunks):,}")
print(f"Laenge Median  : {int(np.median([len(c) for c in chunks]))} Zeichen")
print()
for c in chunks[:3]:
    print("-", c[:160], "...")

## 2. BM25S – term-based

BM25 bewertet, wie oft Query-Terme im Chunk vorkommen, gewichtet nach Seltenheit
des Terms im Korpus (IDF) und normiert auf die Chunk-Laenge.

Der Stemmer ist bei Deutsch wichtig: ohne ihn matcht `Klimaschutz` nicht auf
`Klimaschutzpolitik` und `Steuern` nicht auf `Steuer`.

In [ ]:
import bm25s

try:
    import Stemmer
    stemmer = Stemmer.Stemmer("german")
except ImportError:
    stemmer = None
    print("PyStemmer nicht installiert - laeuft ohne Stemming (schwaechere Treffer)")

corpus_tokens = bm25s.tokenize(chunks, stopwords="de", stemmer=stemmer)

bm25 = bm25s.BM25()
bm25.index(corpus_tokens)
print("BM25-Index gebaut")

In [ ]:
def search_bm25(query: str, k: int = TOP_K):
    """Gibt (indices, scores) der k besten Chunks zurueck."""
    query_tokens = bm25s.tokenize(query, stopwords="de", stemmer=stemmer, show_progress=False)
    idx, scores = bm25.retrieve(query_tokens, k=k, show_progress=False)
    return idx[0], scores[0]


idx, scores = search_bm25("Was ist die Stellungnahme der Partei zum Klimaschutz?")
for i, s in zip(idx, scores):
    print(f"{s:6.2f}  {chunks[i][:140]}")

## 3. Embeddings – dense retrieval

`paraphrase-multilingual-MiniLM-L12-v2` ist auf mehrsprachigen Paaren trainiert und
versteht Deutsch. Mit `normalize_embeddings=True` sind alle Vektoren auf Laenge 1,
dann ist das Skalarprodukt direkt die Kosinus-Aehnlichkeit.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

embeddings = model.encode(
    chunks,
    batch_size=64,
    normalize_embeddings=True,
    show_progress_bar=True,
)
print(embeddings.shape)   # (n_chunks, 384)

In [ ]:
def search_embed(query: str, k: int = TOP_K):
    """Gibt (indices, scores) der k aehnlichsten Chunks zurueck."""
    q = model.encode([query], normalize_embeddings=True)[0]
    sims = embeddings @ q              # Kosinus-Aehnlichkeit, da normalisiert
    idx = np.argsort(-sims)[:k]
    return idx, sims[idx]


idx, scores = search_embed("Was ist die Stellungnahme der Partei zum Klimaschutz?")
for i, s in zip(idx, scores):
    print(f"{s:6.3f}  {chunks[i][:140]}")

## 4. Vergleich

Die Scores der beiden Verfahren sind **nicht** vergleichbar: BM25 ist unbeschraenkt
nach oben, Kosinus liegt zwischen -1 und 1. Vergleichbar ist nur, *welche* Chunks
oben stehen. Dafuer der Overlap: wie viele der Top-k finden beide Verfahren.

In [ ]:
QUERIES = [
    "Was ist die Stellungnahme der Partei zum Klimaschutz?",
    "Wie steht die Partei zur Europaeischen Union?",
    "Welche Position vertritt die Partei bei der Rente?",
    "Zuwanderung und Asylpolitik",
    "Digitalisierung und kuenstliche Intelligenz",
]


def compare(query: str, k: int = TOP_K, width: int = 110) -> pd.DataFrame:
    b_idx, b_scores = search_bm25(query, k)
    e_idx, e_scores = search_embed(query, k)

    return pd.DataFrame({
        "rank":       range(1, k + 1),
        "bm25_score": np.round(b_scores, 2),
        "bm25_chunk": [chunks[i][:width] for i in b_idx],
        "emb_score":  np.round(e_scores, 3),
        "emb_chunk":  [chunks[i][:width] for i in e_idx],
    }).set_index("rank")


pd.set_option("display.max_colwidth", 120)
compare(QUERIES[0])

In [ ]:
rows = []
for q in QUERIES:
    b_idx, _ = search_bm25(q)
    e_idx, _ = search_embed(q)
    shared = set(b_idx.tolist()) & set(e_idx.tolist())
    rows.append({
        "query": q,
        "overlap": len(shared),
        "overlap_pct": f"{len(shared) / TOP_K:.0%}",
    })

pd.DataFrame(rows)

In [ ]:
# Einzelne Query im Detail anschauen
compare(QUERIES[3])

## Was du in den Ergebnissen sehen solltest

BM25 gewinnt, wenn die Query die Begriffe des Texts woertlich enthaelt, und faellt
aus, sobald du umschreibst. Embeddings finden Umschreibungen, holen aber auch
thematisch benachbarte Chunks, die die Frage nicht beantworten.

Der Satz-Split hat zwei bekannte Schwaechen, die du in den Treffern siehst:
Abkuerzungen und Gliederungsnummern (`12.1 Klimaschutzpolitik`) werden
faelschlich getrennt, und ein einzelner Satz ist oft zu wenig Kontext fuer eine
Antwort. Naechster Schritt waere ein Sliding Window ueber je 3 Saetze mit einem
Satz Ueberlappung, dann laesst sich messen, ob die Trefferqualitaet steigt.